<a href="https://colab.research.google.com/github/ShahJahanBrohii/ML-Internship-Flyrank/blob/main/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Title + Abstract

# Which pages should an editor review first?

**Abstract.** Can a learned ranking help an editor choose which keyword articles to inspect first? I used the anonymized FlyRank starter slice of 30,000 content items and compared a transparent rule with Logistic Regression, a Decision Tree, and a constrained Random Forest. On the documented client-holdout split, the Random Forest measured `Precision@50` of 0.740 versus 0.240 for the rule baseline, against an observed decline-label base rate of 0.542. Leakage checks excluded identifiers, trend-derived fields, and product decision flags, while the grouped split held out complete clients. The resulting queue is directional decision support for human review, not evidence that refreshing a page will cause recovery.

## 1. Question

The research question is: **which existing keyword articles should an editor review first, given observed exposure, freshness, position, content depth, and engagement signals?** The output is a ranked queue with reason codes and suggested actions. A reviewer uses it to focus attention; the model does not publish, delete, merge, or rewrite content.

In [3]:
from pathlib import Path
import pandas as pd

repo_candidates = [Path("."), Path("../.."), Path("/content/drive/MyDrive/flyrank-ml-internship-starter")]
repo_root = next((path for path in repo_candidates if (path / "data/raw/content_refresh_anonymized.csv").exists()), None)
if repo_root is None:
    raise FileNotFoundError("Run this notebook from the repository root or clone the repository in Colab.")

paper_artifacts = [
    repo_root / "work/notebooks/w05_model.ipynb",
    repo_root / "work/notebooks/w06_validation_audit.ipynb",
    repo_root / "work/notebooks/w07_action_playbook.ipynb",
    repo_root / "outputs/model_report.md",
]
print("Question: rank observed review opportunities for human editorial triage.")
print(f"Repository root found: {repo_root}")
print(f"Paper source artifacts present: {sum(path.exists() for path in paper_artifacts)}/{len(paper_artifacts)}")

Question: rank observed review opportunities for human editorial triage.
Repository root found: /content/drive/MyDrive/flyrank-ml-internship-starter
Paper source artifacts present: 4/4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Data

This paper uses the bundled anonymized starter export: 30,000 rows, one row per pseudonymized content item, across 32 pseudonymized clients. The lane is restricted to `keyword article`, with trailing 90-day performance fields and content attributes. The broader FlyRank warehouse release contains `dim_clients`, `dim_content`, `fact_content_daily_performance`, `fact_content_daily_performance_sample`, and `fact_content_query_90d`, covering 2025-01-27 to 2026-06-30; this analysis does not claim to benchmark the full approximately 79-million-row release.

Identifiers are used only for grouping and joining. `content_id`, `client_id`, `trend_direction`, `trend_pct`, product scores, product flags, titles, URLs, client names, domains, and raw queries are excluded from model features or public outputs. The target is the observed proxy `trend_direction == "down"`; it is not a later refresh-success outcome.

In [4]:
data_summary = pd.DataFrame([
    {"item": "starter rows", "value": 30000},
    {"item": "pseudonymized clients", "value": 32},
    {"item": "lane", "value": "keyword article"},
    {"item": "metric windows", "value": "trailing 90 days"},
    {"item": "warehouse daily release", "value": "2025-01-27 to 2026-06-30"},
])
display(data_summary)

,item,value
0,starter rows,30000
1,pseudonymized clients,32
2,lane,keyword article
3,metric windows,trailing 90 days
4,warehouse daily release,2025-01-27 to 2026-06-30


## 3. Methodology

The task is ranking, so `Precision@50` is the primary decision metric: how many of the first 50 rows carry the observed positive label? The baseline is a fixed stale-and-visible rule. Candidate models are Logistic Regression, a shallow Decision Tree, and a constrained Random Forest; the Random Forest is selected by held-out ranking performance.

The final feature boundary includes observed age, freshness, search, click, session, position, CTR, engagement, scroll, and content-depth fields. Leakage checks exclude the label and its source fields, identifiers, and any product decision fields. Validation holds out complete clients with seed 42, so the after-split question is whether ranking transfers to clients unseen during training. This is an observed, directional evaluation, not a causal design.

In [5]:
method_checks = pd.DataFrame([
    {"check": "primary metric", "value": "Precision@50"},
    {"check": "split", "value": "client holdout"},
    {"check": "seed", "value": 42},
    {"check": "excluded features", "value": "IDs, trend-derived fields, product decisions"},
    {"check": "claim type", "value": "observed / directional / decision-support"},
])
display(method_checks)

,check,value
0,primary metric,Precision@50
1,split,client holdout
2,seed,42
3,excluded features,"IDs, trend-derived fields, product decisions"
4,claim type,observed / directional / decision-support


## 4. Results (vs baseline)

The comparison is on the same client-holdout design and the same ranking metrics. The Random Forest measured higher ranking separation and top-50 precision than the transparent rule on this starter slice. The base rate remains visible because precision must be interpreted against the 0.542 observed positive rate.

In [6]:
results = pd.DataFrame([
    {"method": "rule baseline", "roc_auc": 0.627, "average_precision": 0.468, "precision_at_50": 0.240},
    {"method": "logistic regression", "roc_auc": 0.700, "average_precision": 0.522, "precision_at_50": 0.400},
    {"method": "decision tree", "roc_auc": 0.742, "average_precision": 0.575, "precision_at_50": 0.540},
    {"method": "random forest", "roc_auc": 0.750, "average_precision": 0.618, "precision_at_50": 0.740},
])
display(results)
print("Observed label base rate: 0.542")
print("The random forest found approximately 37 positives in its first 50 rows; the rule found approximately 12.")

,method,roc_auc,average_precision,precision_at_50
0,rule baseline,0.627,0.468,0.24
1,logistic regression,0.700,0.522,0.40
2,decision tree,0.742,0.575,0.54
3,random forest,0.750,0.618,0.74


Observed label base rate: 0.542
The random forest found approximately 37 positives in its first 50 rows; the rule found approximately 12.


## 5. Limitations & honest framing

This is a 30,000-row anonymized starter slice, not a benchmark on the full warehouse. The observed decline label is derived from trend fields, and the trailing metrics may overlap with how that label was formed; a future-window label with strictly earlier features would support a stronger deployment claim. Client holdout tests unseen-client ranking but does not test every future time period or editorial segment. The score prioritizes review; it does not show that an edit will cause recovery, improve traffic, or create business value.

In [7]:
limitations = [
    "No causal refresh outcome is observed.",
    "The starter slice is not the full warehouse benchmark.",
    "Temporal overlap between trailing features and the observed trend label is a limitation.",
    "Human review is required before any public content change.",
]
for item in limitations:
    print(f"- {item}")

- No causal refresh outcome is observed.
- The starter slice is not the full warehouse benchmark.
- Temporal overlap between trailing features and the observed trend label is a limitation.
- Human review is required before any public content change.


## 6. Ranked recommendations

The action playbook turns the score into a review order:

1. **Refresh review:** inspect high-confidence, visible decline-risk rows first.
2. **CTR review:** check title, metadata, intent, and snippet fit for visible page-one rows with low CTR.
3. **Engagement review:** inspect page experience and intent alignment when sessions exist but engagement signals are weak.
4. **Expand and refresh:** check thin visible pages for missing coverage before editing.
5. **Monitor:** route low-volume, missing-data, or conflicting-signal rows to observation.

No action is automatic. Reviewers verify the page, query intent, data window, editorial context, accessibility, legal context, and reversibility before acting.

In [8]:
action_counts = pd.DataFrame([
    {"suggested_action": "monitor", "rows": 13093},
    {"suggested_action": "refresh", "rows": 8178},
    {"suggested_action": "refresh_and_review_ctr", "rows": 6657},
    {"suggested_action": "refresh_and_review_engagement", "rows": 1990},
    {"suggested_action": "expand_and_refresh", "rows": 82},
])
display(action_counts)
print("These are ranked review recommendations, not automated publishing decisions.")

,suggested_action,rows
0,monitor,13093
1,refresh,8178
2,refresh_and_review_ctr,6657
3,refresh_and_review_engagement,1990
4,expand_and_refresh,82


These are ranked review recommendations, not automated publishing decisions.


## 7. Artifacts the paper embeds

The deployed page reuses the model comparison, action mix, confidence mix, top-feature, and trend-distribution charts. The queue is regenerated by `work/notebooks/w07_action_playbook.ipynb`; row-level queue data stays out of git by design, while metrics JSON receipts and paper figures are committed.

In [9]:
artifact_paths = [
    "docs/index.html",
    "docs/assets/action_mix.svg",
    "docs/assets/confidence_mix.svg",
    "docs/assets/top_feature_importance.svg",
    "docs/assets/trend_distribution.svg",
    "outputs/model_report.md",
    "work/notebooks/w05_model.ipynb",
    "work/notebooks/w06_validation_audit.ipynb",
    "work/notebooks/w07_action_playbook.ipynb",
]
for artifact in artifact_paths:
    print(f"{artifact}: {(repo_root / artifact).exists()}")

docs/index.html: True
docs/assets/action_mix.svg: True
docs/assets/confidence_mix.svg: True
docs/assets/top_feature_importance.svg: True
docs/assets/trend_distribution.svg: True
outputs/model_report.md: True
work/notebooks/w05_model.ipynb: True
work/notebooks/w06_validation_audit.ipynb: True
work/notebooks/w07_action_playbook.ipynb: True


## Self-check

- [x] The paper mirror includes Title + Abstract, Introduction / Problem, Data, Methodology, Results, Limitations, Ranked recommendations, Reproducibility, and Acknowledgments & data credit.
- [x] Model and baseline metrics use the same documented client-holdout design and primary ranking metric.
- [x] Public-safe language is observed, measured, directional, and decision-support; no client names or private queries appear.
- [x] The page is Pages-ready at `docs/index.html` with relative chart assets.
- [ ] Run every cell top to bottom, inspect the rendered page on desktop and mobile, deploy from `/docs`, and record the exact URL in `submission/paper_url.txt`.

## ML-12 closing: 5-minute demo outline

1. **Question:** show the decision and why a review queue is more honest than an automatic fix.
2. **Evidence:** show the client-holdout model-vs-baseline table and the observed base rate.
3. **Audit:** show the leakage exclusions, grouped split, and one failure pattern.
4. **Playbook:** show the action mapping, reason codes, human-review rules, and no-go list.
5. **Close:** show the live paper, reproducibility links, and the limits of the claim.

### Social-post cut

I built a public-safe content refresh ranking study on FlyRank's anonymized starter slice. A client-held-out Random Forest measured 0.740 Precision@50 versus 0.240 for a transparent rule baseline, but the output is only directional decision support for human review, not proof that editing causes recovery. Paper and notebooks: this repository's deployed `/docs` page.

### Employer-facing summary

I built an end-to-end, reproducible ML workflow that ranks content pages for editorial review, compares a learned model with a transparent baseline, and validates the comparison by holding out entire clients. The strongest measured result was 0.740 Precision@50 for a Random Forest versus 0.240 for the baseline on the anonymized starter slice. I also converted the score into an explainable action playbook with leakage checks, failure analysis, monitoring triggers, and explicit no-go automation rules.